# 01 — Ingest

> **AI-Assisted Development** — This project was built with [Kiro](https://kiro.dev). See `README.md` and `SOURCES.md` for full disclosure.

Fetches all raw data for the **dui-by-state** project and loads it into DuckDB.
Raw files are saved to `data/raw/` unchanged. Nothing is modified here.

### Data sources (all U.S. government / official)
| Table | Source | Notes |
|---|---|---|
| `state_ref` | U.S. Census Bureau Gazetteer | Hand-curated CSV — lat/lng, area, region |
| `fips_codes` | U.S. Census Bureau `state.txt` | Official FIPS codes + abbreviations |
| `population` | Census NST-EST2024 | State pop estimates 2020–2024 |
| `fars_trends` | NHTSA FARS 2015–2024 | Alcohol-impaired fatalities by state/year |
| `fars_2024` | NHTSA FARS 2024 | 2024 state summary |
| `dui_laws_ncsl` | NCSL | DUI criminal status by state |
| `dui_penalties_roadlaw` | roadlawguide.com | First-offense penalties (cross-ref) |
| `dui_penalties_ailawyer` | ailawyer.pro | First-offense penalties (cross-ref) |
| `fars_prior_dwi_speed` | NHTSA FARS 2024 | Prior DWI convictions & crash speed |
| `speed_limits_vmt` | IIHS + FHWA | Max speed limits & VMT |
| `states` | Derived | Master reference: all of the above joined |
| `_sources` | Metadata | Provenance record for every table |

Full attribution: see `SOURCES.md`.

---
**To re-run ingestion:** execute `scripts/ingest_all.py` from the project root.
This notebook loads the already-ingested data for inspection and verification.

In [ ]:
import sys
sys.path.insert(0, "..")

import duckdb
import pandas as pd
from pathlib import Path
from src.ingest import load_config

cfg = load_config("../config.yaml")
DB  = Path("..") / cfg["settings"]["duckdb_file"]
con = duckdb.connect(str(DB))

print("Project:", cfg["project_name"])
print("DuckDB: ", DB.resolve())
print()
print("Tables in database:")
con.execute("SHOW TABLES").df()

## Source provenance
Every table has an entry in `_sources` recording where the data came from.

In [ ]:
con.execute("SELECT duckdb_table, source_name, license, retrieved FROM _sources").df()

---
## 1. State reference
Hand-curated from the Census Bureau 2024 Gazetteer.
51 rows (50 states + DC). Includes lat/lng centroids, land area, Census region/division.

In [ ]:
state_ref = con.execute("SELECT * FROM state_ref ORDER BY state_fips").df()
print(f"{len(state_ref)} rows  {list(state_ref.columns)}")
state_ref

---
## 2. Population estimates (Census NST-EST2024)
Annual July 1 estimates per state, 2020–2024.

In [ ]:
pop = con.execute("SELECT * FROM population ORDER BY state_fips").df()
print(f"{len(pop)} rows  {list(pop.columns)}")
pop

---
## 3. FARS — alcohol-impaired fatalities (2015–2024)
Source: NHTSA Fatality Analysis Reporting System.

**Methodology note:**
- 2015–2020: `DRUNK_DR > 0` in `accident.csv` flags alcohol-involved crashes.
- 2021–2024: NHTSA restructured the schema. Alcohol involvement derived from
  `drimpair.csv` where `drimpair = 9` (Under the Influence of Alcohol, Drugs or Medication),
  joined to `accident.csv` on `ST_CASE`.
- The 2021+ figures are lower than 2015–2020 because the new schema captures only
  cases explicitly coded — not all alcohol-involved crashes are coded by officers.
  NHTSA's own published totals (using statistical imputation) are higher.

In [ ]:
# National totals by year
national = con.execute("""
    SELECT
        year,
        SUM(total_fatalities)    AS total_fatalities,
        SUM(alcohol_fatalities)  AS alcohol_fatalities,
        ROUND(SUM(alcohol_fatalities) * 100.0 / SUM(total_fatalities), 1) AS pct_alcohol
    FROM fars_trends
    GROUP BY year
    ORDER BY year
""").df()
national

In [ ]:
# 2024 by state — top 15 by alcohol fatality %
con.execute("""
    SELECT state_fips, state_name, total_fatalities, alcohol_fatalities,
           pct_fatalities_alcohol AS pct_alcohol
    FROM fars_2024
    ORDER BY pct_fatalities_alcohol DESC
    LIMIT 15
""").df()

---
## 4. DUI laws & penalties
Three sources scraped and cross-referenced.
Always verify against official state statutes before publishing.

In [ ]:
# NCSL — DUI criminal status (misdemeanor vs felony thresholds)
ncsl = con.execute("SELECT * FROM dui_laws_ncsl").df()
print(f"NCSL: {len(ncsl)} rows")
ncsl.head(10)

In [ ]:
# roadlawguide — penalties: BAC, fines, jail, suspension, IID
roadlaw = con.execute("SELECT * FROM dui_penalties_roadlaw").df()
print(f"roadlawguide: {len(roadlaw)} rows")
roadlaw.head(10)

In [ ]:
# ailawyer — penalties: jail, fine, suspension, IID, lookback, felony threshold
ail = con.execute("SELECT * FROM dui_penalties_ailawyer").df()
print(f"ailawyer: {len(ail)} rows")
ail.head(10)

---
## 5. Master states table
Joins state reference, population, and FARS 2024 summary into one row per state.

In [ ]:
states = con.execute("SELECT * FROM states ORDER BY state_fips").df()
print(f"{len(states)} rows × {len(states.columns)} cols")
print(f"Columns: {list(states.columns)}")
states

---
## 6. NIAAA Alcohol Consumption by State (2022)
Source: National Institute on Alcohol Abuse and Alcoholism, Surveillance Report #121.

- **URL:** https://www.niaaa.nih.gov/sites/default/files/surveillance-report121.pdf
- **Method:** PDF Table 2 extracted via `pdfplumber`
- **License:** Public domain (U.S. government work)
- **Key field:** `ethanol_per_capita_gallons_2022` — gallons of pure ethanol per person age 14+
- **Decile:** 1 = highest consumption, 10 = lowest. National avg 2.50 gal.

In [ ]:
consumption = con.execute("SELECT * FROM alcohol_consumption ORDER BY ethanol_per_capita_gallons_2022 DESC").df()
print(f"{len(consumption)} states")
consumption

---
## 7. NHTSA Imputed Alcohol Fatalities (2024)
Source: NHTSA Traffic Safety Facts — State Alcohol-Impaired-Driving Estimates, 2024.

- **URL:** https://crashstats.nhtsa.dot.gov/Api/Public/ViewPublication/813813
- **Method:** PDF Table 2 extracted via `pdfplumber`
- **License:** Public domain (U.S. government work)
- **Key difference from `fars_2024`:** These are **statistically imputed** estimates.
  NHTSA uses multiple imputation for untested drivers. National 2024: 11,907 fatalities (30%).
  The raw FARS coding (`fars_2024`) only captures explicitly coded cases (~15%).
  **Use this table for published analysis** — it's what NHTSA publishes and media cite.

In [ ]:
nhtsa = con.execute("SELECT * FROM nhtsa_imputed_2024 ORDER BY pct_alcohol_impaired_2024 DESC").df()
print(f"{len(nhtsa)} states | National: {nhtsa['alcohol_impaired_fatalities_2024'].sum():,} fatalities")
nhtsa

---
## 8. FBI UCR DUI Arrests by State (2023)
Source: FBI Uniform Crime Reporting Program — ICPSR Study 39298.

- **URL:** https://www.icpsr.umich.edu/web/NACJD/studies/39298
- **File:** `data/raw/ICPSR_39298-V1.zip` → `DS0002/39298-0002-Data.tsv`
- **Method:** ⚠️ **Manual download** — requires free ICPSR account registration
- **License:** ICPSR Terms of Use — no redistribution of raw data; derivative analysis OK; cite source
- **Offense code:** 220 = Driving Under the Influence

### ⚠️ Important limitation
Not all law enforcement agencies report to UCR. State-level totals reflect **only reporting
agencies** and significantly undercount actual arrests. Coverage varies widely by state.
Use `reporting_population` to compute per-capita rates for fair comparison.

**Citation:** United States Dept. of Justice, FBI. UCR Program Data: Arrests by Age,
Sex, and Race, Summarized Yearly, 2023. ICPSR [distributor], 2026.

In [ ]:
arrests = con.execute("SELECT * FROM dui_arrests_2023 ORDER BY total_dui_arrests DESC").df()
print(f"{len(arrests)} states | Total reported DUI arrests: {arrests['total_dui_arrests'].sum():,}")
print(f"Reporting agencies: {arrests['reporting_agencies'].sum():,} | Reporting pop: {arrests['reporting_population'].sum():,}")
arrests

---
## 9. FARS Prior DWI & Crash Speed
Source: NHTSA FARS 2024 `vehicle.csv` — `PREV_DWI` and `VSPD_LIM`.
Key: what % of impaired fatal-crash drivers are repeat offenders?

In [ ]:
prior_dwi = con.execute("SELECT * FROM fars_prior_dwi_speed ORDER BY state_fips").df()
print(f"{len(prior_dwi)} states")
print(f"Median % impaired with prior DWI: {prior_dwi['pct_impaired_with_prior_dwi'].median():.1f}%")
prior_dwi[['state_fips','pct_impaired_with_prior_dwi','median_crash_speed_limit','pct_crashes_high_speed']].head(10)

---
## 10. Speed Limits & VMT
- IIHS max posted speed limits (rural interstates, Aug 2026)
- FHWA Highway Statistics 2022 — vehicle miles traveled

In [ ]:
speed_vmt = con.execute("SELECT * FROM speed_limits_vmt ORDER BY state_name").df()
print(f"{len(speed_vmt)} states")
print(f"Speed: {speed_vmt['max_speed_limit_mph'].min()}-{speed_vmt['max_speed_limit_mph'].max()} mph")
print(f"VMT: {speed_vmt['vmt_millions_2022'].min():,}-{speed_vmt['vmt_millions_2022'].max():,}M miles")
speed_vmt.head(10)

---
## Re-running ingestion

If you need to re-pull data from source (e.g., a new year of FARS data):

```bash
cd /path/to/dui-by-state
/opt/anaconda3/envs/data_projects/bin/python scripts/ingest_all.py
```

The script uses `fetch_cached()` — files already on disk won't be re-downloaded.
Delete a specific file from `data/raw/` to force a re-fetch of that source.

---
**Next:** open `02-clean.ipynb` for data cleaning and quality checks.

In [ ]:
con.close()